<a href="https://colab.research.google.com/github/hcy05020-maker/Earth-Engine/blob/GIS/Get_started_with_Earth_Engine_for_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get started with Earth Engine for Python

In [ ]:
#@title Copyright 2024 The Earth Engine Community Authors { display-mode: "form" }
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

This quickstart will give you an interactive introduction to visualizing and
analyzing geospatial data with the Earth Engine Python interface.

## Before you begin

[Register or create](https://code.earthengine.google.com/register) a Google Cloud Project; you'll be prompted to complete the following steps. If you already have a project registered for Earth Engine access, skip to the next section.

  * Select the project's purpose: commercial or noncommercial.
  * If the purpose is noncommercial, select a project type.
  * Create a new Google Cloud project or select an existing project.
  * If the purpose is commercial, verify or set up billing for your project.
  * Confirm your project information.  

**Note:** If you don't plan to keep the resources that you create in this procedure, create a project instead of selecting an existing project. After you finish these steps, you can [delete the project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects), removing all resources owned by the project.

## Notebook setup

**1.** Import the Earth Engine and geemap libraries.

In [22]:
import ee
import geemap

**2.** Authenticate and initialize the Earth Engine service. Follow the
resulting prompts to complete authentication. Be sure to replace PROJECT_ID
with the name of the project you set up for this quickstart.

In [4]:
ee.Authenticate()
ee.Initialize(project='my-project-495906')

## Add raster data to a map

**1.** Load climate data for a given period and display its metadata.

In [5]:
jan_2023_climate = (
    ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
    .filterDate('2023-01', '2023-02')
    .first()
)
jan_2023_climate

**2.** Instantiate a map object and add the temperature band as a layer with
specific visualization properties. Display the map.

In [6]:
m = geemap.Map(center=[30, 0], zoom=2)

vis_params = {
    'bands': ['temperature_2m'],
    'min': 229,
    'max': 304,
    'palette': 'inferno',
}
m.add_layer(jan_2023_climate, vis_params, 'Temperature (K)')
m

Map(center=[30, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', trans…

## Add vector data to a map

**1.** Create a vector data object with points for three cities.

In [12]:
cities = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(128.28, 37.88), {'city': 'Gangwon-do'}),
    ee.Feature(ee.Geometry.Point(128.7, 36.3), {'city': 'Gyeongsangbuk-do'}),
    ee.Feature(ee.Geometry.Point(128.69, 35.23), {'city': 'Gyeongsangnam-do'}),
])
cities

**2.** Add the city locations to the map and redisplay it.

In [8]:
m.add_layer(cities, name='Cities')
m

Map(bottom=6702.0, center=[36.56260003738548, 127.01141532290985], controls=(WidgetControl(options=['position'…

## Extract and chart data

**1.** Import the Altair charting library.

In [9]:
%pip install -q --upgrade altair
import altair as alt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 10.3 MB/s eta 0:00:00


**2.** Extract the climate data for the three cities as a pandas DataFrame.

In [10]:
city_climates = jan_2023_climate.reduceRegions(cities, ee.Reducer.first())

city_climates_dataframe = ee.data.computeFeatures(
    {'expression': city_climates, 'fileFormat': 'PANDAS_DATAFRAME'}
)
city_climates_dataframe

,geo,city,dewpoint_temperature_2m,dewpoint_temperature_2m_max,dewpoint_temperature_2m_min,evaporation_from_bare_soil_max,evaporation_from_bare_soil_min,evaporation_from_bare_soil_sum,evaporation_from_open_water_surfaces_excluding_oceans_max,evaporation_from_open_water_surfaces_excluding_oceans_min,...,volumetric_soil_water_layer_1_min,volumetric_soil_water_layer_2,volumetric_soil_water_layer_2_max,volumetric_soil_water_layer_2_min,volumetric_soil_water_layer_3,volumetric_soil_water_layer_3_max,volumetric_soil_water_layer_3_min,volumetric_soil_water_layer_4,volumetric_soil_water_layer_4_max,volumetric_soil_water_layer_4_min
0,"{'type': 'Point', 'coordinates': [128.28, 37.88]}",Gangwon-do,259.904283,277.600479,239.747940,0,-0.000005,-0.000052,0.000008,-0.000002,...,0.361725,0.358473,0.413620,0.331192,0.337032,0.345413,0.326004,0.382088,0.384842,0.379944
1,"{'type': 'Point', 'coordinates': [128.7, 36.3]}",Gyeongsangbuk-do,263.165760,285.749374,242.769714,0,-0.000119,-0.001224,0.000063,-0.000020,...,0.317657,0.345106,0.388962,0.316376,0.316484,0.329956,0.304962,0.322966,0.323196,0.322845
2,"{'type': 'Point', 'coordinates': [128.69, 35.23]}",Gyeongsangnam-do,266.843626,287.424698,247.851273,0,-0.000168,-0.017369,0.000018,-0.000026,...,0.280792,0.332751,0.417938,0.289291,0.313203,0.325745,0.297806,0.328397,0.329330,0.327927


**3.** Plot the temperature for the cities as a bar chart.

In [11]:
alt.Chart(city_climates_dataframe).mark_bar(size=100).encode(
    alt.X('city:N', sort='y', axis=alt.Axis(labelAngle=0), title='City'),
    alt.Y('temperature_2m:Q', title='Temperature (K)'),
    tooltip=[
        alt.Tooltip('city:N', title='City'),
        alt.Tooltip('temperature_2m:Q', title='Temperature (K)'),
    ],
).properties(title='January 2023 temperature for selected cities', width=500)

alt.Chart(...)

## What's next

  * Learn about analyzing data with Earth Engine's [objects and methods](https://developers.google.com/earth-engine/guides/objects_methods_overview).
  * Learn about Earth Engine's [processing environments](https://developers.google.com/earth-engine/guides/processing_environments).
  * Learn about Earth Engine's [machine learning capabilities](https://developers.google.com/earth-engine/guides/machine-learning).
  * Learn how to [export your computation results to BigQuery](https://developers.google.com/earth-engine/guides/exporting_to_bigquery).

## 산불 피해지 GIS 분석: 피해 강도 및 경사도

이 섹션에서는 한국의 대형 산불 피해지를 분석하여 산불 피해 강도(dNBR)와 경사도 레이어를 포함하는 GIS를 구축합니다. 2022년 울진-삼척 산불을 예시로 들어 진행합니다.

In [13]:
# 1. 산불 지역 및 기간 정의 (2022년 울진-삼척 산불 예시)

# 울진-삼척 산불 대략적인 관심 지역(ROI) 정의
# (경도, 위도) 순서로 Point 또는 Polygon을 정의합니다.
fire_roi = ee.Geometry.Polygon([
  [129.1, 36.8],
  [129.5, 36.8],
  [129.5, 37.1],
  [129.1, 37.1],
  [129.1, 36.8]
]);

# 산불 발생 전/후 기간 정의
pre_fire_start = '2022-02-01'
pre_fire_end = '2022-02-28' # 산불 전 (2월)

post_fire_start = '2022-03-01' # 산불 발생 3월 4일경
post_fire_end = '2022-04-30' # 산불 후 (3~4월)

print(f"분석 지역: {fire_roi.getInfo()}")
print(f"산불 전 기간: {pre_fire_start} ~ {pre_fire_end}")
print(f"산불 후 기간: {post_fire_start} ~ {post_fire_end}")

분석 지역: {'type': 'Polygon', 'coordinates': [[[129.1, 36.8], [129.5, 36.8], [129.5, 37.1], [129.1, 37.1], [129.1, 36.8]]]}
산불 전 기간: 2022-02-01 ~ 2022-02-28
산불 후 기간: 2022-03-01 ~ 2022-04-30


In [21]:
# 2. 위성 영상 확보 및 전처리

# Sentinel-2 Level-2A (Surface Reflectance) 영상 컬렉션 정의
# DeprecationWarning에 따라 COPERNICUS/S2_SR_HARMONIZED 사용
S2_SR_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'

# Sentinel-2 구름 마스크 함수 정의
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000) # SR 값을 0-1 범위로 스케일링

# NBR(Normalized Burn Ratio) 계산 함수 정의
def calculate_nbr(image):
    # NIR (B8), SWIR2 (B12) 밴드를 사용하여 NBR 계산
    # NBR = (NIR - SWIR2) / (NIR + SWIR2)
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

In [18]:
# 산불 전 영상 컬렉션 필터링 및 처리
pre_fire_images = (ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(pre_fire_start, pre_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # 구름 비율 20% 미만 필터링
    .map(mask_s2_clouds) # 구름 마스크 적용
    .map(calculate_nbr)) # NBR 계산

# 가장 구름 없는 산불 전 영상 선택 (중간값)
pre_fire_image = pre_fire_images.median().clip(fire_roi)

# 산불 후 영상 컬렉션 필터링 및 처리
post_fire_images = (ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(post_fire_start, post_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # 구름 비율 20% 미만 필터링
    .map(mask_s2_clouds) # 구름 마스크 적용
    .map(calculate_nbr)) # NBR 계산

# 가장 구름 없는 산불 후 영상 선택 (중간값)
post_fire_image = post_fire_images.median().clip(fire_roi)

print("산불 전/후 Sentinel-2 영상 및 NBR 계산 완료.")

산불 전/후 Sentinel-2 영상 및 NBR 계산 완료.


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/S2_SR_HARMONIZED

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


In [19]:
# 3. 산불 피해 강도(dNBR) 계산

# NBR 밴드가 존재하는지 확인 후 계산
if pre_fire_image.bandNames().contains('NBR') and post_fire_image.bandNames().contains('NBR'):
    # dNBR = NBR_pre - NBR_post
    dnbr = pre_fire_image.select('NBR').subtract(post_fire_image.select('NBR')).rename('dNBR')
    print("dNBR 계산 완료.")
else:
    dnbr = ee.Image(0).rename('dNBR') # NBR 밴드가 없으면 0으로 초기화 (오류 방지)
    print("경고: NBR 밴드를 찾을 수 없어 dNBR 계산을 건너뜁니다.")

# 4. 경사도(Slope) 레이어 생성

# SRTM Digital Elevation Model (DEM) 데이터 로드
dem = ee.Image('USGS/SRTMGL1_003').clip(fire_roi)

# 경사도 계산 (도 단위)
slope = ee.Terrain.slope(dem).rename('Slope_Degrees')

print("경사도 레이어 계산 완료.")

dNBR 계산 완료.
경사도 레이어 계산 완료.


In [23]:
# 5. GIS 레이어 시각화

# 지도 객체 생성
m = geemap.Map(center=[37.0, 129.3], zoom=9) # 울진-삼척 지역 중심

# dNBR 시각화 파라미터 (일반적인 산불 피해 강도 분류)
# dNBR 값은 보통 -1 ~ 1 사이이며, 양수 값이 클수록 피해가 큼
dnbr_vis_params = {
    'min': -0.5,
    'max': 1.0,
    'palette': [
        '#006400',  # Green (무피해/식생증가)
        '#00FF00',  # Light Green (낮은 피해)
        '#FFFF00',  # Yellow (중간-낮은 피해)
        '#FFA500',  # Orange (중간-높은 피해)
        '#FF0000',  # Red (높은 피해)
        '#8B0000'   # Dark Red (심각한 피해)
    ]
}

# 경사도 시각화 파라미터
slope_vis_params = {
    'min': 0,
    'max': 45, # 최대 45도까지 표시
    'palette': ['lightblue', 'blue', 'darkblue'] # 경사가 높을수록 어두운 파랑
}

# 레이어를 지도에 추가
m.add_layer(pre_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Pre-Fire (RGB)')
m.add_layer(post_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Post-Fire (RGB)')
m.add_layer(dnbr, dnbr_vis_params, 'Burn Severity (dNBR)')
m.add_layer(slope, slope_vis_params, 'Slope')
m.add_colorbar(dnbr_vis_params, label="Burn Severity (dNBR)", orientation='vertical')
m.add_colorbar(slope_vis_params, label="Slope (Degrees)", orientation='vertical')

# 지도 표시
m

Map(center=[37.0, 129.3], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…